# Stage 1: Extract Beat-Level Arrhythmia Features Per Patient



In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path("..").resolve()

# Folder you downloaded from PhysioNet — see DATA.md
ANNOTATION_DIR = BASE_DIR / "external_data" / "vitaldb-arrhythmia-database-1.0.0" / "Annotation_Files"

# The clinical file that tells us which 477 caseids to process
CLINICAL_CSV = BASE_DIR / "data" / "interim" / "imputed_477_cases.csv"

# Draft output — verified separately in 05_verify_arrhythmia_features.ipynb before being
# promoted to the final arrhythmia_features_482.csv
DRAFT_CSV = BASE_DIR / "data" / "interim" / "arrhythmia_features_draft.csv"

print(f"Annotation folder exists: {ANNOTATION_DIR.exists()}")
print(f"Clinical CSV exists: {CLINICAL_CSV.exists()}")

In [ ]:
# Tunable constants used throughout the extraction
MIN_CLEAN_ROWS = 10     # below this many clean rows, give up on the patient entirely
RR_MIN_SEC = 0.2        # RR intervals shorter than this are physiologically impossible (artifacts)
RR_MAX_SEC = 3.0        # RR intervals longer than this are also treated as artifacts
MIN_VALID_RR = 3        # need at least this many valid RR intervals to compute rr_cv/rr_mean

## Data Quality Finding (Read Before the Extraction Function)

While testing this extraction, we discovered that `beat_type` can be missing (`NaN`) even on rows that pass the `bad_signal_quality == False` filter. Across all 482 annotation files, 1,148 such "clean but unlabeled" rows exist. Most (991/1148) occur during segments where `rhythm_label == 'Noise'` — the rhythm itself was too noisy to classify a beat type for, even though the row wasn't flagged `bad_signal_quality`. A smaller number (157/1148) occur scattered within otherwise normally-labeled segments, for no single clear reason.

**Why this matters:** the spec's Step 2 compares `beat_type != 'N'` to find the first non-normal beat. In pandas, `NaN != 'N'` evaluates to `True` — so without explicitly excluding `NaN` first, every one of these unlabeled rows would be wrongly counted as "the patient's arrhythmia event," which could badly corrupt `event_time_sec` for the affected patients. The code below explicitly filters out `NaN` `beat_type` rows before that comparison.

**Why Check 5 will still show some patients outside its 0.95–1.05 tolerance:** per the spec, `total_beats` counts *every* clean row, including these unlabeled ones, but they never get counted into `n_normal`/`n_supraventricular`/`n_ventricular`. For patients with many such rows (worst case: 137 of 1,462 clean rows in one file, ~9.4%), `pct_normal + pct_supraventricular + pct_ventricular` falls noticeably below 1.0 — not because of a counting bug, but because a real chunk of that patient's clean beats have no usable type. This is expected and is explained in the verification notebook when it happens, rather than treated as an error.

In [ ]:
def extract_features(case_id):
    """
    Implements Steps 1-5 of the spec for one patient's annotation file.
    Always returns a dict with all 13 keys (caseid + 12 features) so the
    final dataframe has a consistent shape even for patients where
    extraction fails.
    """
    # A row of all-NaN values, used whenever extraction can't proceed
    empty = {
        "caseid": case_id, "event_time_sec": np.nan, "rhythm_onset_normal": np.nan,
        "arrhythmia_duration_sec": np.nan, "total_beats": np.nan, "n_normal": np.nan,
        "n_supraventricular": np.nan, "n_ventricular": np.nan, "pct_normal": np.nan,
        "pct_supraventricular": np.nan, "pct_ventricular": np.nan, "rr_cv": np.nan, "rr_mean": np.nan,
    }

    ann_path = ANNOTATION_DIR / f"Annotation_file_{case_id}.csv"
    if not ann_path.exists():
        print(f"WARNING: annotation file missing for caseid {case_id} - recording as NaN")
        return empty

    ann = pd.read_csv(ann_path)

    # ---- STEP 1: quality filter ----
    clean = ann[ann["bad_signal_quality"] == False].copy()
    if len(clean) < MIN_CLEAN_ROWS:
        print(f"WARNING: caseid {case_id} has only {len(clean)} clean rows "
              f"(< {MIN_CLEAN_ROWS}) - recording as NaN, no further extraction attempted")
        return empty

    # Sort by time so "earliest"/"latest" and the RR-interval differencing below are correct
    clean = clean.sort_values("time_second")

    # ---- STEP 2: event timestamp ----
    # .notna() guards against the NaN-beat_type rows described above leaking into
    # the "non-normal" set via the NaN != 'N' quirk
    non_normal = clean[clean["beat_type"].notna() & (clean["beat_type"] != "N")]

    if len(non_normal) > 0:
        event_time_sec = non_normal["time_second"].min()
        rhythm_onset_normal = False
    else:
        # No non-normal beats survived quality filtering - this patient's annotated
        # segment is entirely Normal Sinus Rhythm. Use the very start of the clip instead.
        event_time_sec = clean["time_second"].min()
        rhythm_onset_normal = True

    # ---- STEP 3: arrhythmia duration ----
    if len(non_normal) > 0:
        arrhythmia_duration_sec = non_normal["time_second"].max() - non_normal["time_second"].min()
    else:
        arrhythmia_duration_sec = 0.0

    # ---- STEP 4: beat composition ----
    # total_beats counts EVERY clean row, including ones with an unusable (NaN) beat_type
    # (see the data quality note above - this is why pct_normal+pct_svt+pct_vt can be < 1)
    total_beats = len(clean)
    n_normal = (clean["beat_type"] == "N").sum()
    n_supraventricular = (clean["beat_type"] == "S").sum()
    n_ventricular = (clean["beat_type"] == "V").sum()
    pct_normal = n_normal / total_beats
    pct_supraventricular = n_supraventricular / total_beats
    pct_ventricular = n_ventricular / total_beats

    # ---- STEP 5: RR interval variability ----
    times_sorted = clean["time_second"].values  # already sorted above
    rr_intervals = np.diff(times_sorted)         # consecutive time differences
    valid_rr = rr_intervals[(rr_intervals >= RR_MIN_SEC) & (rr_intervals <= RR_MAX_SEC)]
    if len(valid_rr) >= MIN_VALID_RR:
        # np.std default (ddof=0, population standard deviation) - the spec didn't specify
        # sample vs. population, so we use numpy's plain default here
        rr_cv = np.std(valid_rr) / np.mean(valid_rr)
        rr_mean = np.mean(valid_rr)
    else:
        rr_cv = np.nan
        rr_mean = np.nan

    return {
        "caseid": case_id,
        "event_time_sec": event_time_sec,
        "rhythm_onset_normal": rhythm_onset_normal,
        "arrhythmia_duration_sec": arrhythmia_duration_sec,
        "total_beats": total_beats,
        "n_normal": n_normal,
        "n_supraventricular": n_supraventricular,
        "n_ventricular": n_ventricular,
        "pct_normal": pct_normal,
        "pct_supraventricular": pct_supraventricular,
        "pct_ventricular": pct_ventricular,
        "rr_cv": rr_cv,
        "rr_mean": rr_mean,
    }


# Quick sanity check on case 337, the case we've used as a running example all along
print("Case 337:", extract_features(337))

In [ ]:
clinical = pd.read_csv(CLINICAL_CSV)
case_ids = clinical["caseid"].tolist()
print(f"Processing {len(case_ids)} case_ids from {CLINICAL_CSV}")

results = [extract_features(cid) for cid in case_ids]
features_df = pd.DataFrame(results)

print(f"\nfeatures_df shape: {features_df.shape}")
print(features_df.head())

In [ ]:
assert features_df["caseid"].is_unique, "caseid should be unique"

features_df.to_csv(DRAFT_CSV, index=False)
print(f"Saved DRAFT (unverified) features to {DRAFT_CSV.resolve()}")
print("Run verify_arrhythmia_features.ipynb next to check this draft before it becomes final.")